In [1]:
import re
import os
import time
import json
import uuid
import calendar
from dateutil import parser
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor

import requests
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt
from datetime import datetime, timezone
from scipy.interpolate import RBFInterpolator
from scipy.interpolate import PchipInterpolator
from collections import deque

from api_client import TradingDeskAPI
from options import OptionSurface, Deribit, OKX, Bybit
from scanner import MarketScanner
from portfolio_management import Portfolio

In [2]:
load_dotenv(r"D:/OneDrive/Trading/Prediction Markets/Moreton Capital/.env")
BASE_URL = os.getenv("BASE_URL")
USER_EMAIL = os.getenv("USER_EMAIL")
USER_PASSWORD = os.getenv("USER_PASSWORD")
JWT_TOKEN = os.getenv("JWT")
print(BASE_URL)
portfolio = Portfolio()
target_expiry_str = "25DEC26"
target_expiry = datetime.strptime(target_expiry_str, "%d%b%y").strftime("%Y%m%d")

currencies = ["BTC", "ETH"]

s = OptionSurface()

deribit = Deribit(currencies=currencies, target_expiry=target_expiry)
okx = OKX(currencies=currencies, target_expiry=target_expiry)
bybit = Bybit(currencies=currencies, target_expiry=target_expiry)

s.initialize(currencies=currencies, exchanges=[deribit, okx, bybit])

https://alphasignal-dev.moretoncp.com
spot: 62769.0 volume24h: 3.0198
spot: 1874.5 volume24h: 54.3529
spot: 62766.6 volume24h: 7259.4480232
spot: 1872.31 volume24h: 45202.738625
spot: 62769.1 volume24h: 6769.476543
spot: 1872.1 volume24h: 53778.34233


In [3]:
api = TradingDeskAPI(base_url=BASE_URL, email=USER_EMAIL, password=USER_PASSWORD, token=JWT_TOKEN)

markets = api.get_markets(limit=10000, liquidity_num_min=10000, volume_num_min=5000)

all_markets_df = pd.DataFrame(markets)

all_markets_df.head()

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,clobRewards,oneWeekPriceChange,oneMonthPriceChange,oneYearPriceChange,seriesColor,showGmpSeries,showGmpOutcome,marketMetadata,line,umaResolutionStatus
0,3511089,LoL: MVK Esports vs CTBC Flying Oyster (BO5) -...,0xd3622a52ec6d9a8d582d0aae715c0b2a0b15e72a67cc...,lol-mvk-cfo-2026-08-14,https://gol.gg/esports/home,2026-08-14T15:15:00Z,99778.8501,2026-08-11T10:31:17Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,562828,"2026 Balance of Power: D Senate, D House",0x16c63b7cc37f012b9f59ee164ec03877914c701d06d4...,2026-balance-of-power-d-senate-d-house-949,,2026-11-03T00:00:00Z,99607.1393,2025-07-11T21:05:18.843Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,"[{'id': '632970', 'conditionId': '0x16c63b7cc3...",0.04,0.050,0.27,,False,False,NaN,NaN,NaN
2,601823,Will Eduardo Bolsonaro win the 2026 Brazilian ...,0xe1d5733322fd2215f136412f419d9ff805d097f78c68...,will-eduardo-bolsonaro-win-the-2026-brazilian-...,,2026-10-04T00:00:00Z,993196.87586,2025-09-18T20:08:02.206Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,-0.001,NaN,,False,False,NaN,NaN,NaN
3,665374,Will the U.S. invade Iran before 2027?,0x5db999fad322cea2914535aae5517060c3f80ad6d8c0...,will-the-us-invade-iran-before-2027,,2026-12-31T00:00:00Z,992919.0051,2025-11-05T17:52:17.414Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,"[{'id': '288102', 'conditionId': '0x5db999fad3...",NaN,-0.010,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1235563,Will the New York Mets win the 2026 World Series?,0x98b595f40feccf49ded360d1eb15b9bb81041cc6006d...,will-the-new-york-mets-win-the-2026-world-series,,2026-10-31T23:55:00Z,99130.81549,2026-01-21T20:45:12.574Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,-0.005,NaN,,False,False,NaN,NaN,NaN


In [ ]:
# api = TradingDeskAPI(base_url=BASE_URL, email=USER_EMAIL, password=USER_PASSWORD, token=JWT_TOKEN)
# markets1 = api.list_markets(limit=500)
# all_markets_df1 = pd.DataFrame(markets1)

# all_markets_df1["liquidityNum"] = pd.to_numeric(all_markets_df1["liquidityNum"], errors="coerce")
# all_markets_df1["volumeNum"] = pd.to_numeric(all_markets_df1["volumeNum"], errors="coerce")

# all_markets_df1 = all_markets_df1[(all_markets_df1["liquidityNum"] >= 10000) & (all_markets_df1["volumeNum"] >= 5000)]
# all_markets_df1 = all_markets_df1.sort_values("liquidityNum", ascending=False).reset_index(drop=True)

# all_markets_df1.head()

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,makerBaseFee,takerBaseFee,gameId,sportsMarketType,eventStartTime,feeSchedule,oneHourPriceChange,clobRewards,oneYearPriceChange,marketMetadata
0,561249,Will Greg Abbott win the 2028 US Presidential ...,0x01dffa7abae7e5d9b7fb44b06d537c5ac932e2ca422a...,will-greg-abbott-win-the-2028-us-presidential-...,,2028-11-07T00:00:00Z,2290255.79328,2025-07-11T19:06:08.361Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,1000.0,1000.0,NaN,NaN,NaN,"{'exponent': 1, 'rate': 0.04, 'takerOnly': Tru...",NaN,NaN,0.005,NaN
1,559665,Will Cory Booker win the 2028 Democratic presi...,0x1970bcce75e3674660917ae4685b433a9b1e0152d810...,will-corey-booker-win-the-2028-democratic-pres...,,2028-11-07T00:00:00Z,2018415.08907,2025-07-11T18:36:10.848Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,1000.0,1000.0,NaN,NaN,NaN,"{'exponent': 1, 'rate': 0.04, 'takerOnly': Tru...",NaN,NaN,-0.008,NaN
2,3128885,Strait of Hormuz traffic returns to normal by ...,0x49479ffa88e2897569b1d413e3bc558003e21912edee...,strait-of-hormuz-traffic-returns-to-normal-by-...,,2026-08-15T00:00:00Z,1851330.89078,2026-07-27T17:50:28.17351Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2252246,Will the Fed increase interest rates by 50+ bp...,0x2e4b58fc18dbffd74d5275d89fb076943f21992763c4...,will-the-fed-increase-interest-rates-by-50-bps...,NaN,2026-09-16T00:00:00Z,1014072.16326,2026-05-13T21:23:16.501737Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,1000.0,1000.0,NaN,NaN,NaN,"{'exponent': 1, 'rate': 0.05, 'takerOnly': Tru...",NaN,NaN,NaN,NaN
4,3563239,Dota 2: Team Yandex vs Team Liquid (BO3) - The...,0x3b7971cd72e15ff80e623a7521fa3e0545b296545792...,dota2-ty-liquid-2026-08-13,https://www.dotabuff.com,2026-08-14T08:00:00Z,786995.05327,2026-08-13T11:19:35Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,1000.0,1000.0,1633343,moneyline,2026-08-14T02:00:00Z,"{'exponent': 1, 'rate': 0.05, 'takerOnly': Tru...",NaN,NaN,NaN,NaN


In [4]:
BTC_KEYWORDS = [
    "bitcoin",
    "btc",
    "xbt"
]

ETH_KEYWORDS = [
    "ethereum",
    " eth "
]

KEYWORDS = [
    "bitcoin",
    "btc",
    "xbt",
    "ethereum",
    " eth "
]

scanner = MarketScanner(api=api, BASE_URL=BASE_URL, USER_EMAIL=USER_EMAIL, USER_PASSWORD=USER_PASSWORD)
markets_df, opportunities_df = scanner.scan_market(markets_df=all_markets_df, s=s, KEYWORDS=KEYWORDS)

opportunities_df

iv 0.5604465790288528
p_touch_above: 1.0 p_touch_below: 0.02082

BTC
buy_yes_ev: 0.0058537200000000015
sell_yes_ev: -0.008718170000000003
buy_no_ev: -0.008718170000000022
sell_no_ev: 0.005853720000000003
buy_yes_kelly: 0.0029713297530565763
sell_yes_kelly: -0.3602004820758515
buy_no_kelly: -0.3602004820758528
sell_no_kelly: 0.002971329753056577
iv 0.4858952153189467
p_touch_above: 0.00255 p_touch_below: 1.0

BTC
buy_yes_ev: -0.0017288800000000003
sell_yes_ev: 0.00024062999999999992
buy_no_ev: 0.00024063000000001554
sell_no_ev: -0.0017288799999999847
buy_yes_kelly: -0.0008681547299107205
sell_yes_kelly: 0.043113920512572415
buy_no_kelly: 0.04311392051257496
sell_no_kelly: -0.0008681547299107127
iv 0.6133530705614699
p_touch_above: 1.0 p_touch_below: 0.0

BTC
buy_yes_ev: -0.02030473
sell_yes_ev: 0.01676268
buy_no_ev: 0.01676268000000003
sell_no_ev: -0.020304729999999993
buy_yes_kelly: -0.010362778417823739
sell_yes_kelly: 0.5
buy_no_kelly: 0.5
sell_no_kelly: -0.010362778417823735
iv 0.58

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,buy_yes_ev,sell_yes_ev,buy_no_ev,sell_no_ev,buy_yes_kelly,sell_yes_kelly,buy_no_kelly,sell_no_kelly,best_ev,best_action
54,701486,"Will Bitcoin reach $200,000 by December 31, 2026?",0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,will-bitcoin-reach-200000-by-december-31-2026-...,,2027-01-01T05:00:00Z,92222.04015,2025-11-24T19:07:20.933Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,-0.020622,0.016945,0.016945,-0.020622,-0.010536,0.478808,0.478808,-0.010536,0.016945,buy_no_ev
40,1343228,"Will Bitcoin dip to $5,000 by December 31, 2026?",0xe681a6326237f3b17ce8622728b6cd104281dfe4d2b3...,will-bitcoin-dip-to-5000-by-december-31-2026-j...,,2027-01-01T05:00:00Z,94381.40138,2026-02-05T22:18:33.034Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,-0.020305,0.016763,0.016763,-0.020305,-0.010363,0.500000,0.500000,-0.010363,0.016763,buy_no_ev
82,701554,"Will Ethereum dip to $800 by December 31, 2026?",0x717672a48c5f2938631f6e467b9a48aedb03fa380c01...,will-ethereum-dip-to-800-by-december-31-2026-568,,2027-01-01T05:00:00Z,89701.3851,2025-11-24T19:27:14.172Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0.008068,-0.027777,-0.027777,0.008068,0.004409,-0.212223,-0.212223,0.004409,0.008068,buy_yes_ev
7,3257364,"Will Bitcoin dip to $47,500 in August?",0xda345549bd2bc7738321d3b4eefa07d9545748a8a133...,will-bitcoin-dip-to-47pt5k-in-august-2026,NaN,2026-09-01T04:00:00Z,98342.46613,2026-08-01T05:07:36Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0.005854,-0.008718,-0.008718,0.005854,0.002971,-0.360200,-0.360200,0.002971,0.005854,sell_no_ev
55,3257342,"Will Bitcoin reach $75,000 in August?",0xc4bde1d397625e1990fec7f787b9eb2e5727c666d048...,will-bitcoin-reach-75k-in-august-2026,NaN,2026-09-01T04:00:00Z,92200.78617,2026-08-01T05:07:38Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,-0.007752,0.004075,0.004075,-0.007752,-0.003961,0.115151,0.115151,-0.003961,0.004075,buy_no_ev
12,3257334,"Will Bitcoin reach $85,000 in August?",0x7f668805bfee7909997562f60489ddef01e40a3dca43...,will-bitcoin-reach-85k-in-august-2026,NaN,2026-09-01T04:00:00Z,97694.86839,2026-08-01T05:07:39Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,-0.001729,0.000241,0.000241,-0.001729,-0.000868,0.043114,0.043114,-0.000868,0.000241,buy_no_ev


In [5]:
# get all dfs
DATA_DIR = f"data"

orders_df = pd.read_parquet(f"{DATA_DIR}/orders.parquet")
fills_df = pd.read_parquet(f"{DATA_DIR}/fills.parquet")
positions_df = pd.read_parquet(f"{DATA_DIR}/positions.parquet")
realized_pnl_df = pd.read_parquet(f"{DATA_DIR}/realized_pnl.parquet")
equity_df = pd.read_parquet(f"{DATA_DIR}/equity.parquet")

# fills_df = fills_df.iloc[0:0]

In [ ]:
# DATA_DIR = f"data"
# os.makedirs(DATA_DIR, exist_ok=True)
# filename = f"{DATA_DIR}/fills.parquet"
# fills_df.to_parquet(filename, engine="fastparquet", index=False)

In [ ]:
fills_df = portfolio.sync_fills(api, fills_df)

{'id': '4c3c68e2-7b89-463c-ae03-48b92808eeef', 'taker_order_id': '0x8039083a81a0749075a21658567ad6e0e449ea2ac99edb17d0053452709e311a', 'market': '0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f3a0e74426f0762e668cd', 'asset_id': '96993471854400156408670527613150944443359272190785251193551242374636006072800', 'side': 'BUY', 'size': '4.87', 'fee_rate_bps': '0', 'price': '0.981', 'status': 'CONFIRMED', 'match_time': '1786695911', 'last_update': '1786695921', 'outcome': 'No', 'bucket_index': 0, 'owner': '6912b6a4-d0bf-e1c2-717b-363c17380f43', 'maker_address': '0x4b5D4502ae3cd3d5a090A12e7eca0D8b0146d471', 'transaction_hash': '0x72d7e7cf667485ea8622a7afe382ad092edace5637a6fe38f5a3d1257d7cf776', 'maker_orders': [{'order_id': '0xdd2798ece1ab952dd962049ac36e11eafc00d88d11aba3be8af2c32bab321e82', 'owner': 'f78c8482-7fa0-b0a3-c013-4c3281eb3fa2', 'maker_address': '0x07AE5837E8a3fb4e45Bc0B5eCF9b085B9733A897', 'matched_amount': '4.87', 'price': '0.981', 'fee_rate_bps': '', 'asset_id': '969934718544001

In [ ]:
orders_df = portfolio.sync_orders(api, orders_df)

In [ ]:
def manage_open_orders(api, orders_df, markets_df):

    def fee(price, fee_rate):
        #fee = C × feeRate × p × (1 - p)
        #Where C = number of shares traded and p = price of the shares.
        return fee_rate * price * (1 - price)

    api_orders = api.list_orders()

    for order in api_orders["orders"]:

        # Example:
        # cancel if the order is too old
        # cancel if the price is no longer competitive
        # cancel if the EV has disappeared
        # cancel if the market has changed
        # cancel if position/inventory limits have changed

        side = order["side"]
        token_id = order["asset_id"]

        row = markets_df[(markets_df["yes_token"] == token_id) | (markets_df["no_token"] == token_id)].iloc[0]
        is_yes = token_id == row["yes_token"]
        fee_rate = float(row["feeSchedule"]["rate"])

        p_yes = row["model_prob"]
        p_no = 1 - p_yes

        if side == "BUY":
            if is_yes:
                buy_yes_cost = order["price"] + fee(order["price"], fee_rate)
            
                profit_if_yes = 1 - buy_yes_cost
                cost_if_no = buy_yes_cost
    
                # EV: profit if yes - cost if no
                buy_yes_ev = p_yes * profit_if_yes - p_no * cost_if_no

                should_cancel = buy_yes_ev < ENTRY_EV_THRESHOLD

            else:
                buy_no_cost =  order["price"] + fee( order["price"], fee_rate)
                
                profit_if_no = 1 - buy_no_cost
                cost_if_yes = buy_no_cost
    
                buy_no_ev = p_no * profit_if_no - p_yes * cost_if_yes

                should_cancel = buy_no_ev < ENTRY_EV_THRESHOLD


        elif side == "SELL":
            if is_yes:
                hold_ev = p_yes
                exit_ev = order["price"] - fee(order["price"], fee_rate)

            else:
                hold_ev = p_no
                exit_ev = order["price"] - fee(order["price"], fee_rate)

            should_cancel = (exit_ev - hold_ev) < EXIT_EV_THRESHOLD

        else:
            should_cancel = False


        if should_cancel:

            confirm_order = (input("Place this order? Type YES to confirm: ") == "YES")

            if confirm_order == True:
                try:
                    api.cancel_order(order["order_id"])

                    orders_df.loc[orders_df["order_id"] == order["order_id"], "status"] = "CANCELLED"
                    orders_df.loc[orders_df["order_id"] == order["order_id"], "cancelled_at"] = datetime.now(timezone.utc)

                    orders_df["cancelled_at"] = pd.to_datetime(orders_df["cancelled_at"], utc=True)

                    print("\nCANCEL ORDER SUBMITTED\n\n")

                except Exception as e:
                    print(f"ORDER CANCELLATION ERROR: {e}")

            else:
                print("skipping cancel order")

    return orders_df

In [15]:
api.list_orders()

{'orders': [], 'total': 0}

In [ ]:
orders_df = manage_open_orders(api, orders_df, markets_df)

In [10]:
# Inventory Management Step

EXIT_EV_THRESHOLD = 0.02

def run_risk_management(positions_df, markets_df, orders_df):
    positions = api.get_positions(positions_df, markets_df)
    print(positions)
    # positions =
    # [
    #     {
    #         "condition_id": "0x7b9072e6...",
    #         "token_id": "123456789...",
    #         "outcome": "Yes",
    #         "shares": 25.0
    #     },
    #     {
    #         "condition_id": "0xdaa4866b...",
    #         "token_id": "987654321...",
    #         "outcome": "No",
    #         "shares": 10.0
    #     }
    # ]

    def fee(price, fee_rate):
        #fee = C × feeRate × p × (1 - p)
        #Where C = number of shares traded and p = price of the shares.
        return fee_rate * price * (1 - price)

    for pos in positions:
        print("pos:", pos)

        condition_id = pos["condition_id"]
        row = markets_df.loc[markets_df["conditionId"] == condition_id].iloc[0]
        print("row: ", row)
        fee_rate = float(row["feeSchedule"]["rate"])
        p_yes = row["model_prob"]
        p_no = 1 - p_yes

        if pos["outcome"] == "Yes": # no shorting in polymarket
            current_bid = row["yes_bid"]
            hold_ev = p_yes
            exit_ev = current_bid - fee(current_bid, fee_rate)

        elif pos["outcome"] == "No":
            current_bid = row["no_bid"]
            hold_ev = p_no
            exit_ev = current_bid - fee(current_bid, fee_rate)

        if (exit_ev - hold_ev) > EXIT_EV_THRESHOLD:
            best_action = "exit"

        else:
            best_action = "hold"

        size = pos["shares"]
        client_oid = str(uuid.uuid4())

        portfolio.format_signal_exit(best_action, row, pos, current_bid, client_oid, hold_ev, exit_ev, positions_df, EXIT_EV_THRESHOLD)

        if best_action == "exit":
            confirm_order = (input("Place this order? Type YES to confirm: ") == "YES")

            if confirm_order == True:

                price_tick = float(api.tick_size(pos["token_id"])["tick_size"])
                price = api.round_to_tick(current_bid, price_tick)
                normalized_size = api.normalize_size(size)
                
                try:
                    order = api.place_limit_order_test(
                        token_id=pos["token_id"],
                        side="sell",
                        price=price,
                        size=normalized_size,
                        order_type="GTC",
                    )
            
                    print("\nLIMIT ORDER SUBMITTED\n\n")
                    print(order)
            
                    order_row = {
                        "order_id": order["clob_order_id"],
                        "condition_id": condition_id,
                        "token_id": pos["token_id"],
                        "outcome": pos["outcome"],
                        "side": "sell",
                        "price": price,
                        "requested_size": normalized_size,
                        "order_type": "GTC",
                        "status": "OPEN",
                        "created_at": datetime.now(timezone.utc),
                        "cancelled_at": None,
                    }

                    orders_df = pd.concat([orders_df, pd.DataFrame([order_row])], ignore_index=True)
                    orders_df["created_at"] = pd.to_datetime(orders_df["created_at"], utc=True)
                    orders_df["cancelled_at"] = pd.to_datetime(orders_df["cancelled_at"], utc=True)
            
                    print("\nLIMIT ORDER RECORDED\n\n")
        
                except Exception as e:
                    print("\nLIMIT ORDER ERROR\n\n")
                    print(e)
                    break

            else:
                print("\nSKIPPING LIMIT ORDER\n\n")

    return orders_df

orders_df = run_risk_management(positions_df, markets_df, orders_df)

[{'condition_id': '0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f3a0e74426f0762e668cd', 'token_id': '96993471854400156408670527613150944443359272190785251193551242374636006072800', 'question': 'Will Bitcoin reach $200,000 by December 31, 2026?', 'outcome': 'No', 'shares': 4.87}]
pos: {'condition_id': '0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f3a0e74426f0762e668cd', 'token_id': '96993471854400156408670527613150944443359272190785251193551242374636006072800', 'question': 'Will Bitcoin reach $200,000 by December 31, 2026?', 'outcome': 'No', 'shares': 4.87}
row:  id                                                             701486
question            Will Bitcoin reach $200,000 by December 31, 2026?
conditionId         0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...
slug                will-bitcoin-reach-200000-by-december-31-2026-...
resolutionSource                                                     
                                          ...                        
sell_yes_ke

In [ ]:
# New Opportunities Step

ENTRY_EV_THRESHOLD = 0.01   # require 1% edge, default = 0
MAX_POSITION = 0.05
FRACTION = 0.1

def run_new_opportunities(opportunities_df, orders_df):

    cash = float(api.balance()["balance"])
    # cash = 100
    
    for _, trade in opportunities_df.iterrows():

        if trade["best_ev"] < ENTRY_EV_THRESHOLD:
            print(f"Current trade is less than required ev ({ENTRY_EV_THRESHOLD}), skipping")
            continue

        # usually cannot short, this only happens when im closing positions
        if trade["best_action"] == "buy_yes_ev" or trade["best_action"] == "sell_no_ev":
            best_action = "buy_yes_ev"
            ev = trade["buy_yes_ev"]
            token_id = trade["yes_token"]
            outcome = "YES"
            kelly = trade["buy_yes_kelly"]
            current_ask = trade["yes_ask"]

        elif trade["best_action"] == "buy_no_ev" or trade["best_action"] == "sell_yes_ev":
            best_action = "buy_no_ev"
            ev = trade["buy_no_ev"]
            token_id = trade["no_token"]
            outcome = "NO"
            kelly = trade["buy_no_kelly"]
            current_ask = trade["no_ask"]

        print(cash, FRACTION, kelly, MAX_POSITION)

        dollars = min(cash * FRACTION * kelly, cash * MAX_POSITION)

        current_balance = api.balance(asset_type="conditional", token_id=token_id) # TO BE CHANGED
        current_size = float(current_balance["balance"])
        print("current_balance", current_balance)
        print("current balance held: ", current_size)
        # current_size = 0

        # inventory cap
        dollars = min(dollars, cash * MAX_POSITION - current_size * current_ask)

        if dollars <= 0:
            print("no more dollars to allocate for this trade position, skipping\n")
            continue

        if dollars < 1.0:
            print("Order too small after position cap, skipping")
            continue

        # min order amount
        dollars = max(1.0, dollars)
        print("min order cap...")

        size = dollars / current_ask

        normalized_size = api.normalize_size(size)

        unrealized_pnl = normalized_size * ev
        client_oid = str(uuid.uuid4())

        portfolio.format_signal_entry(trade, outcome, best_action, normalized_size, current_ask,
                        token_id, client_oid, unrealized_pnl, cash, ENTRY_EV_THRESHOLD, MAX_POSITION)

        confirm_order = (input("Place this order? Type YES to confirm: ") == "YES")
        
        if confirm_order == True:

            # price_tick = api.get_tick_size(token_id)
            price_tick = float(api.tick_size(token_id)["tick_size"])
            price = api.round_to_tick(current_ask, price_tick)

            try:
                order = api.place_limit_order(
                    token_id=token_id,
                    side="buy",
                    price=price,
                    size=normalized_size,
                    order_type="GTC",
                    client_order_id=client_oid
                )
        
                print("\nLIMIT ORDER SUBMITTED\n\n")
                print(order)

                order_row = {
                    "order_id": order["clob_order_id"],
                    "condition_id": trade["conditionId"],
                    "token_id": token_id,
                    "outcome": outcome,
                    "side": "buy",
                    "price": price,
                    "requested_size": normalized_size,
                    "order_type": "GTC",
                    "status": "OPEN",
                    "created_at": datetime.now(timezone.utc),
                    "cancelled_at": None,
                }
        
                orders_df = pd.concat([orders_df, pd.DataFrame([order_row])], ignore_index=True)
                orders_df["created_at"] = pd.to_datetime(orders_df["created_at"], utc=True)
                orders_df["cancelled_at"] = pd.to_datetime(orders_df["cancelled_at"], utc=True)
        
                print("\nLIMIT ORDER RECORDED\n\n")

            except Exception as e:
                print("\nLIMIT ORDER ERROR\n\n")
                print(e)
                break

        else:
            print("\nLIMIT SKIPPING ORDER\n\n")

    return orders_df

orders_df = run_new_opportunities(opportunities_df, orders_df)

100.0 0.1 0.0246208651399491 0.05
current_balance {'balance': '0', 'allowances': {'exchange_v2': True, 'neg_risk_adapter': True, 'neg_risk_exchange_v2': True}, 'approved': True}
current balance held:  0.0
Order too small after position cap, skipping
100.0 0.1 0.47852533473634484 0.05
current_balance {'balance': '0', 'allowances': {'exchange_v2': True, 'neg_risk_adapter': True, 'neg_risk_exchange_v2': True}, 'approved': True}
current balance held:  0.0
min order cap...
BUY SIGNAL
Market                    : Will Bitcoin reach $200,000 by December 31, 2026?
Outcome                   : NO
Direction                 : up
Event Type                : touch
Best Action               : buy_no_ev
Recommended Size          : 4.877934
Current Ask               : 0.981
P Yes                     : 0.00076
P No                      : 0.99924
Buy Yes EV                : -0.020612000000000002
Sell No EV                : -0.020612000000000057
Buy No EV                 : 0.01693527000000004
Sell Yes EV  

In [116]:
realized_pnl_df

,condition_id,token_id,outcome,realized_pnl,realized_shares,realized_fees


In [ ]:
ENTRY_EV_THRESHOLD = 0.01   # require 1% edge, default = 0
EXIT_EV_THRESHOLD = 0.02
FRACTION = 0.25
MAX_POSITION = 0.05

load_dotenv(r"D:/OneDrive/Trading/Prediction Markets/Moreton Capital/.env")
BASE_URL = os.getenv("BASE_URL")
USER_EMAIL = os.getenv("USER_EMAIL")
USER_PASSWORD = os.getenv("USER_PASSWORD")
JWT_TOKEN = os.getenv("JWT")
print(BASE_URL)

portfolio = Portfolio()
api = TradingDeskAPI(base_url=BASE_URL, email=USER_EMAIL, password=USER_PASSWORD, token=JWT_TOKEN)
s = OptionSurface()
scanner = MarketScanner(api=api, BASE_URL=BASE_URL, USER_EMAIL=USER_EMAIL, USER_PASSWORD=USER_PASSWORD)

target_expiry_str = "25DEC26"
target_expiry = datetime.strptime(target_expiry_str, "%d%b%y").strftime("%Y%m%d")

currencies = ["BTC", "ETH"]

KEYWORDS = [
    "bitcoin",
    "btc",
    "xbt",
    "ethereum",
    " eth "
]

DATA_DIR = f"data"

for i in range(1):
    # get all dfs
    orders_df = pd.read_parquet(f"{DATA_DIR}/orders.parquet")
    fills_df = pd.read_parquet(f"{DATA_DIR}/fills.parquet")
    positions_df = pd.read_parquet(f"{DATA_DIR}/positions.parquet")
    realized_pnl_df = pd.read_parquet(f"{DATA_DIR}/realized_pnl.parquet")
    equity_df = pd.read_parquet(f"{DATA_DIR}/equity.parquet")

    # 1. Initialize variance surface
    deribit = Deribit(currencies=currencies, target_expiry=target_expiry)
    okx = OKX(currencies=currencies, target_expiry=target_expiry)
    bybit = Bybit(currencies=currencies, target_expiry=target_expiry)

    s.initialize(currencies=currencies, exchanges=[deribit, okx, bybit])

    # 2. Get current markets
    markets = api.get_markets(limit=10000, liquidity_num_min=10000, volume_num_min=5000)
    all_markets_df = pd.DataFrame(markets)

    # 3. Scan markets
    markets_df, opportunities_df = scanner.scan_market(markets_df=all_markets_df, s=s, KEYWORDS=KEYWORDS)

    # 4. Get newly executed trades
    fills_df = sync_fills(api, fills_df)

    # 5. Reconstruct portfolio
    # 6. Calculate realized P&L
    positions_df, realized_df = portfolio.reconstruct_positions_fifo(fills_df)

    # 7. Get latest order state
    orders_df = sync_orders(api, orders_df)

    # 8. manage cancel orders
    orders_df = manage_open_orders(api, orders_df, markets_df)

    # 9. Mark positions to market
    positions_df = portfolio.mark_positions_to_market(positions_df, markets_df)

    # 10. Calculate equity
    equity_df = portfolio.calculate_equity(api, positions_df, realized_df, equity_df)

    # 11. Risk management
    orders_df = run_risk_management(positions_df, markets_df, orders_df)

    # 12. Risk management
    orders_df = run_new_opportunities(opportunities_df, orders_df)

    portfolio.save_snapshots(markets_df, "markets")
    portfolio.save_snapshots(opportunities_df, "opportunities")
    portfolio.save(orders_df, "orders")
    portfolio.save(fills_df, "fills")
    portfolio.save(positions_df, "positions")
    portfolio.save(realized_pnl_df, "realized_pnl")
    portfolio.save(equity_df, "equity")

    # time.sleep(300)

https://alphasignal-dev.moretoncp.com
spot: 63004.0 volume24h: 2.2643
spot: 1876.0 volume24h: 29.679
spot: 63028.4 volume24h: 6339.59523918
spot: 1875.49 volume24h: 43634.368194
spot: 63017.6 volume24h: 6276.339928
spot: 1875.08 volume24h: 48566.85138
iv 0.490069817899321
p_touch_above: 1.0 p_touch_below: 0.29701

BTC
buy_yes_ev: 0.033885
sell_yes_ev: -0.069778
buy_no_ev: -0.069778
sell_no_ev: 0.03388499999999994
buy_yes_kelly: 0.022992366412213742
sell_yes_kelly: -0.15353911420926633
buy_no_kelly: -0.15353911420926633
sell_no_kelly: 0.022992366412213704
iv 0.4856712939900793
p_touch_above: 0.00327 p_touch_below: 1.0

BTC
buy_yes_ev: -0.0010088800000000006
sell_yes_ev: -0.0004793699999999998
buy_no_ev: -0.00047936999999998375
sell_no_ev: -0.001008879999999985
buy_yes_kelly: -0.0005066077136136273
sell_yes_kelly: -0.08588920781328943
buy_no_kelly: -0.08588920781328606
sell_no_kelly: -0.0005066077136136195
iv 0.6583336891155444
p_touch_above: 1.0 p_touch_below: 0.0

BTC
buy_yes_ev: -0.02

KeyboardInterrupt: 

In [85]:
portfolio.save(orders_df, "orders")

save df saved at:  data/orders.parquet


In [74]:
orders_df

,order_id,condition_id,token_id,outcome,side,price,requested_size,order_type,status,created_at,cancelled_at
0,0x8039083a81a0749075a21658567ad6e0e449ea2ac99e...,0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,NO,buy,0.981,4.877934,GTC,OPEN,2026-08-14 09:21:24.060645+00:00,NaT


In [125]:
portfolio.save(positions_df, "positions")
portfolio.save(realized_pnl_df, "realized_pnl")

save df saved at:  data/positions.parquet
save df saved at:  data/realized_pnl.parquet


In [117]:
fills_df = sync_fills(api, fills_df)

{'id': '4c3c68e2-7b89-463c-ae03-48b92808eeef', 'taker_order_id': '0x8039083a81a0749075a21658567ad6e0e449ea2ac99edb17d0053452709e311a', 'market': '0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f3a0e74426f0762e668cd', 'asset_id': '96993471854400156408670527613150944443359272190785251193551242374636006072800', 'side': 'BUY', 'size': '4.87', 'fee_rate_bps': '0', 'price': '0.981', 'status': 'CONFIRMED', 'match_time': '1786695911', 'last_update': '1786695921', 'outcome': 'No', 'bucket_index': 0, 'owner': '6912b6a4-d0bf-e1c2-717b-363c17380f43', 'maker_address': '0x4b5D4502ae3cd3d5a090A12e7eca0D8b0146d471', 'transaction_hash': '0x72d7e7cf667485ea8622a7afe382ad092edace5637a6fe38f5a3d1257d7cf776', 'maker_orders': [{'order_id': '0xdd2798ece1ab952dd962049ac36e11eafc00d88d11aba3be8af2c32bab321e82', 'owner': 'f78c8482-7fa0-b0a3-c013-4c3281eb3fa2', 'maker_address': '0x07AE5837E8a3fb4e45Bc0B5eCF9b085B9733A897', 'matched_amount': '4.87', 'price': '0.981', 'fee_rate_bps': '', 'asset_id': '969934718544001

In [118]:
positions_df, realized_df = portfolio.reconstruct_positions_fifo(fills_df)

In [119]:
positions_df

,condition_id,token_id,outcome,shares,cost_basis,avg_entry_price,realized_pnl,realized_shares,realized_fees
0,0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,4.87,4.77747,0.981,0.0,0.0,0.0


In [120]:
positions_df = portfolio.mark_positions_to_market(positions_df, markets_df)

In [121]:
equity_df = portfolio.calculate_equity(api, positions_df, realized_df, equity_df)

Equity:  99.99
Return:  -0.0112%
Sharpe: N/A
Sortino: N/A


In [122]:
equity_df

,timestamp,cash,market_value,equity,realized_pnl,unrealized_pnl,daily_return,period_return,sharpe,sortino
0,2026-08-14 07:47:24.047885+00:00,100.00000,0.0000,100.00000,0.0,0.00000,NaN,None,NaN,NaN
1,2026-08-14 09:38:24.441930+00:00,95.21618,4.7726,99.98878,0.0,-0.00487,NaN,-0.000112,NaN,NaN


In [123]:
positions_df

,condition_id,token_id,outcome,shares,cost_basis,avg_entry_price,realized_pnl,realized_shares,realized_fees,current_price,market_value,unrealized_pnl,unrealized_return
0,0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,4.87,4.77747,0.981,0.0,0.0,0.0,0.98,4.7726,-0.00487,-0.001019


In [50]:
realized_df

,condition_id,token_id,outcome,realized_shares,realized_pnl,realized_fees
0,0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,0.0,0.0,0.0


In [15]:
portfolio.save(orders_df, "orders")

save df saved at:  data/orders.parquet


In [ ]:
#                  Polymarket    Model (exchanges: deribit, bybit, okx)    Edge

# 80k Dec-26        22%          20%       +2%
# 90k Dec-26        14%          12%       +2%
# 100k Dec-26        9.5%         5.9%     +3.6%
# 110k Dec-26        7%           4%       +3%
# 120k Dec-26        5%           2%       +3%

In [ ]:
# Keep trading journal

# Your thesis.
# Why you think the market is mispriced.
# Position size.
# Exit criteria.
# What actually happened.

In [ ]:
# Stage 2 — Semi-automated execution

# The bot does:

# find opportunities
# calculate size
# prepare orders

# You approve:

# BUY YES
# Market: BTC above $150k
# Price: 0.43
# Size: $500
# Expected edge: +8%

# Click "confirm".

# This is useful because prediction markets can have:

# ambiguous wording
# resolution risks
# sudden news events

In [ ]:
# Day 1 — API connection + market ingestion

# Goal:

# Can I pull markets automatically?

# Build:

# get_markets()

# Output:

# {
#  "condition_id": "...",
#  "question": "Will X happen?",
#  "tokens": [
#     {
#       "token_id": "YES",
#       "price": 0.42
#     },
#     {
#       "token_id": "NO",
#       "price": 0.58
#     }
#  ]
# }

# Store:

# markets

# condition_id
# question
# yes_token
# no_token
# created_time
# Day 2 — Historical snapshots

# Goal:

# Can I reconstruct what the market looked like yesterday?

# Every 5 minutes:

# while True:

#     markets = api.get_markets()

#     for m in markets:
#         database.save_snapshot(m)

#     sleep(300)

# Database:

# market_snapshots

# timestamp
# condition_id
# yes_price
# no_price
# volume
# Day 3 — Order book collection

# Now collect microstructure data.

# For selected markets:

# get_orderbook(token_id)

# Store:

# orderbook_snapshots

# timestamp

# token_id

# best_bid
# best_ask

# bid_depth
# ask_depth

# Calculate:

# Spread
# spread=ask−bid

# Example:

# Bid:
# 0.42

# Ask:
# 0.46

# Spread:
# 4 cents
# Day 4 — Build your scanner

# Your first scanner should be dumb but useful.

# Signal 1: Large moves
# if abs(price_change_24h) > 0.10:
#     flag()

# Example:

# AI model release

# Yesterday:
# 35%

# Today:
# 52%

# Move:
# +17%
# Signal 2: Liquidity opportunities
# if spread > 0.08:
#     flag()
# Signal 3: Volume spikes
# if volume_today > 5 * average_volume:
#     flag()

# Your output:

# TOP MARKETS TO REVIEW

# 1.
# Question:
# Will Fed cut rates?

# Price:
# 42%

# 24h move:
# +12%

# Reason:
# Large movement


# 2.
# Question:
# Will Company X acquire Y?

# Price:
# 33%

# Spread:
# 11 cents

# Reason:
# Wide market
# Day 5 — Add your probability workflow

# Do NOT automate this yet.

# Create a manual table:

# trade_journal.csv

# market,current_price,my_probability,edge,reason
# Fed cut,0.42,0.55,0.13,"Inflation falling"
# AI launch,0.35,0.45,0.10,"Company comments"

# The key question:

# Market probability:
# 42%

# My probability:
# 55%

# Difference:
# +13%
# Day 6 — Paper trading

# Before your C++ engine touches anything:

# Create:

# paper_buy(
#     market,
#     price,
#     size
# )

# Track:

# Position:
# YES Fed cut

# Entry:
# 42c

# Size:
# $100

# Current:
# 48c

# P&L:
# +$14
# Day 7 — Analytics

# Calculate:

# Return
# profit / capital
# Drawdown

# Largest loss from peak.

# Sortino

# Track:

# returns
# negative returns only

# Your output:

# Paper Portfolio

# Trades:
# 18

# Win rate:
# 61%

# Return:
# +8.4%

# Max drawdown:
# -2.1%

# Sortino:
# 2.4

In [ ]:
{'id': '701496', 'question': 'Will Bitcoin reach $100,000 by December 31, 2026?', 'conditionId': '0xdaa4866bae18be58c5a79d2aeeffd035ec78f1bb49dbd88f72993997778a990f', 'slug': 'will-bitcoin-reach-100000-by-december-31-2026-571-361-361', 
 'resolutionSource': '', 'endDate': '2027-01-01T05:00:00Z', 'liquidity': '94083.1351', 'startDate': '2025-11-24T19:07:17.691Z', 'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 
 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 
'description': 'This market will immediately resolve to "Yes" if any Binance 1 minute candle for Bitcoin (BTC/USDT) between November 24, 2025, 14:00 and December 31, 2026, 23:59 in the ET timezone has a final "High" price equal to or greater than the price specified in the title. Otherwise, this market will resolve to "No."\n\nThe resolution source for this market is Binance, specifically the BTC/USDT "High" prices available at https://www.binance.com/en/trade/BTC_USDT, with the chart settings on "1m" for one-minute candles selected on the top bar.\n\nPlease note that the outcome of this market depends solely on the price data from the Binance BTC/USDT trading pair. Prices from other exchanges, different trading pairs, or spot markets will not be considered for the resolution of this market.', 
'outcomes': '["Yes", "No"]', 'outcomePrices': '["0.095", "0.905"]', 'volume': '2370982.5583320004', 'active': True, 'closed': False, 'marketMakerAddress': '', 'createdAt': '2025-11-24T18:55:12.725029Z', 'updatedAt': '2026-08-02T10:54:52.526927Z', 
'new': False, 'featured': False, 'submitted_by': '0x91430CaD2d3975766499717fA0D66A78D814E5c5', 'archived': False, 'resolvedBy': '0x65070BE91477460D8A7AeEb94ef92fe056C2f2A7', 'restricted': True, 'groupItemTitle': '↑ 100,000', 'groupItemThreshold': '13', 
'questionID': '0x3c9be67d4b90291760ac3bffc1f9470a1966e5c1f3e99131333170e3469bd023', 'enableOrderBook': True, 'orderPriceMinTickSize': 0.01, 'orderMinSize': 5, 'volumeNum': 2370982.5583320004, 'liquidityNum': 94083.1351, 'endDateIso': '2027-01-01', 
'startDateIso': '2025-11-24', 'hasReviewedDates': True, 'volume24hr': 1365.610655, 'volume1wk': 66640.791618, 'volume1mo': 228733.97684700004, 'volume1yr': 2370982.5583320004, 
'clobTokenIds': '["56078938060096976448086754249497300447360333783952000147427828224794011030104", "11291662904897713174667903388388696640643610556195928998276904135282270136756"]', 
'comboStatus': 'disabled', 'umaBond': '500', 'umaReward': '5', 'volume24hrClob': 1365.610655, 'volume1wkClob': 66640.791618, 'volume1moClob': 228733.97684700004, 'volume1yrClob': 2370982.5583320004, 'volumeClob': 2370982.5583320004, 
'liquidityClob': 94083.1351, 'makerBaseFee': 1000, 'takerBaseFee': 1000, 'customLiveness': 0, 'acceptingOrders': True, 'negRisk': False, 'negRiskRequestID': '', 
'events': [{'id': '89502', 'ticker': 'what-price-will-bitcoin-hit-before-2027', 'slug': 'what-price-will-bitcoin-hit-before-2027', 
            'title': 'What price will Bitcoin hit in 2026?', 'description': 'What price will Bitcoin hit before 2027?  ', 
            'resolutionSource': '', 'startDate': '2025-11-24T19:07:12.848Z', 'creationDate': '2025-11-24T19:13:13.705687Z', 'endDate': '2027-01-01T05:00:00Z', 
            'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 
            'active': True, 'closed': False, 'archived': False, 'new': False, 'featured': False, 'restricted': True, 'liquidity': 2695251.24076, 'volume': 50935482.262915, 
            'openInterest': 9503925.568983998, 'createdAt': '2025-11-24T18:55:05.597959Z', 'updatedAt': '2026-08-02T10:55:09.654318Z', 'competitive': 0.9999750006249843, 
            'volume24hr': 130499.32620200001, 'volume1wk': 2062901.9714630004, 'volume1mo': 7099616.726362999, 'volume1yr': 49102712.92022599, 'enableOrderBook': True, 
            'liquidityClob': 2695251.24076, 'negRisk': False, 'commentCount': 0, 'series': [{'id': '10016', 'ticker': 'bitcoin-hit-price-monthly', 'slug': 'bitcoin-hit-price-monthly', 
                                                                                        'title': 'Bitcoin Hit Price Monthly', 'seriesType': 'single', 'recurrence': 'monthly', 
                                                                                        'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/bitcoin+colors.jpeg', 
                                                                                        'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/bitcoin+colors.jpeg', 
                                                                                        'active': True, 'closed': False, 'archived': False, 'featured': False, 'restricted': True, 
                                                                                        'createdAt': '2025-01-31T22:03:50.00441Z', 'updatedAt': '2026-08-02T10:55:28.185077Z', 
                                                                                        'volume24hr': 601000.637578, 'volume': 51492794.133888, 'liquidity': 3438609.2653, 'commentCount': 6318, 
                                                                                        'requiresTranslation': False}], 
            'cyom': False, 'showAllOutcomes': True, 'showMarketImages': False, 'enableNegRisk': False, 'automaticallyActive': True, 'seriesSlug': 'bitcoin-hit-price-monthly', 
            'gmpChartMode': 'default', 'negRiskAugmented': False, 'estimateValue': True, 'cantEstimate': True, 'cumulativeMarkets': False, 'pendingDeployment': False, 'deploying': False, 
            'requiresTranslation': False, 'eventMetadata': {'context_requires_regen': True}, 'version': 'v1'}], 

'ready': False, 'funded': False, 'acceptingOrdersTimestamp': '2025-11-24T19:06:55Z', 
'cyom': False, 'competitive': 0.8590880780051975, 'pagerDutyNotificationEnabled': False, 'approved': True, 'clobRewards': [{'id': '418394', 'conditionId': '0xdaa4866bae18be58c5a79d2aeeffd035ec78f1bb49dbd88f72993997778a990f', 
                                                                                                                        'assetAddress': '0xc011a7e12a19f7b1f670d46f03b03f3342e82dfb', 'rewardsAmount': 0, 'rewardsDailyRate': 0.001, 
                                                                                                                        'startDate': '2026-06-03', 'endDate': '2500-12-31'}], 
'rewardsMinSize': 0, 'rewardsMaxSpread': 0, 'spread': 0.01, 'oneMonthPriceChange': -0.01, 'lastTradePrice': 0.09, 'bestBid': 0.09, 'bestAsk': 0.1, 'automaticallyActive': True, 
'clearBookOnStart': True, 'seriesColor': '', 'showGmpSeries': False, 'showGmpOutcome': False, 'manualActivation': False, 'negRiskOther': False, 'umaResolutionStatuses': '[]', 
'pendingDeployment': False, 'deploying': False, 'deployingTimestamp': '2025-11-24T19:06:23.727362Z', 'rfqEnabled': False, 'holdingRewardsEnabled': True, 'feesEnabled': True, 
'requiresTranslation': False, 'feeType': 'crypto_fees_v2', 'feeSchedule': {'exponent': 1, 'rate': 0.07, 'takerOnly': True, 'rebateRate': 0.2}, 'version': 'v1'}
["0.095", "0.905"]